# OSS-Instruct Pipeline from Magicoder Code Snippets

This notebook keeps **Magicoder responses/code snippets** as the response source, but generates **new instructions** with your chosen OpenAI model so the task-construction process is more comparable to your self-instruct pipelines.

In [ ]:
# =========================
# 1. Install dependencies
# =========================

!pip -q install -U openai datasets pandas tqdm jsonlines huggingface_hub


In [ ]:
# =========================
# 2. Imports and Drive
# =========================

from google.colab import drive, userdata
drive.mount("/content/drive")

import ast
import hashlib
import json
import math
import re
from pathlib import Path
from typing import Any, Dict, List, Optional

import jsonlines
import pandas as pd
from tqdm.auto import tqdm
from datasets import load_dataset
from openai import OpenAI


In [ ]:
# =========================
# 3. Config
# =========================

TARGET_N = 20_000
MAX_SNIPPETS_FOR_GENERATION = 25_000  # keep a buffer before final dedup
HF_TOKEN = userdata.get("HF_TOKEN")
OPENAI_API_KEY = userdata.get("openai_api_key")

if not OPENAI_API_KEY:
    raise ValueError("Missing openai_api_key in Colab secrets.")

client = OpenAI(api_key=OPENAI_API_KEY)

# Dataset source
HF_DATASET_NAME = "ise-uiuc/Magicoder-OSS-Instruct-75K-Instruction-Response"
HF_SPLIT = "train"
LANGUAGE = "python"
USE_STREAMING = False

# OpenAI model settings
MODEL_TASKS = "gpt-4.1-mini"
TASK_TEMPERATURE = 0.7

# Paths
BASE_DIR = Path("/content/drive/MyDrive/Speciale/Data generation/OSS instruct")
BASE_DIR.mkdir(parents=True, exist_ok=True)

OUT_DIR = BASE_DIR / "oss_instruct_from_magicoder_snippets"
OUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_SNIPPETS_PATH = OUT_DIR / "01_raw_python_snippets.jsonl"
FILTERED_SNIPPETS_PATH = OUT_DIR / "02_filtered_python_snippets.jsonl"

TASK_REQUESTS_PATH = OUT_DIR / "03_instruction_requests.jsonl"
TASK_BATCH_META_PATH = OUT_DIR / "04_instruction_batch_meta.json"
TASK_OUTPUTS_PATH = OUT_DIR / "05_instruction_outputs.jsonl"
TASK_ERRORS_PATH = OUT_DIR / "05_instruction_errors.jsonl"

GENERATED_DATASET_PATH = OUT_DIR / "06_generated_instruction_pairs.jsonl"
FINAL_DATASET_PATH = OUT_DIR / "07_oss_instruct_final.jsonl"
REJECTS_PATH = OUT_DIR / "rejects.jsonl"

# Filtering rules for code snippets
MIN_CODE_CHARS = 20
MAX_CODE_CHARS = 8000
MIN_LINES = 1
MAX_LINES = 200


## 4. Pipeline overview

This version uses the **Magicoder code snippet** as the response/target, but generates a **new instruction** with your own OpenAI model.

Flow:
1. Load the Magicoder instruction-response dataset
2. Keep only the Python **response/code** field
3. Clean and filter the code snippets
4. Build one OpenAI Batch request per snippet to generate a new instruction
5. Parse the batch output and pair each new instruction with the original code snippet
6. Deduplicate and export the final training set


In [ ]:
# =========================
# 5. Helper functions
# =========================

def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

def normalize_whitespace(text: str) -> str:
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = text.replace("\t", "    ")
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def strip_code_fences(text: str) -> str:
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```[a-zA-Z0-9_+-]*\n?", "", text)
        text = re.sub(r"\n?```$", "", text)
    return text.strip()

def extract_python_code(text: str) -> str:
    text = normalize_whitespace(text)

    py_blocks = re.findall(r"```python\s*(.*?)```", text, flags=re.IGNORECASE | re.DOTALL)
    if py_blocks:
        return normalize_whitespace(py_blocks[0])

    any_blocks = re.findall(r"```(?:[a-zA-Z0-9_+-]+)?\s*(.*?)```", text, flags=re.DOTALL)
    if any_blocks:
        return normalize_whitespace(any_blocks[0])

    text = re.sub(r"^Here(?:'s| is) .*?:\n", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^Solution:\n", "", text, flags=re.IGNORECASE)
    return strip_code_fences(text)

def parseable_python(code: str) -> bool:
    try:
        ast.parse(code)
        return True
    except Exception:
        return False

def trivial_or_low_value(code: str) -> bool:
    stripped = code.strip()
    if len(stripped) < MIN_CODE_CHARS or len(stripped) > MAX_CODE_CHARS:
        return True

    line_count = len(stripped.splitlines())
    if line_count < MIN_LINES or line_count > MAX_LINES:
        return True

    nonempty = [ln.strip() for ln in stripped.splitlines() if ln.strip()]
    if not nonempty:
        return True

    if all(ln.startswith("import ") or ln.startswith("from ") for ln in nonempty):
        return True
    if all(ln.startswith("#") for ln in nonempty):
        return True
    if re.search(r"(^|\n)\s*def\s+test_", stripped):
        return True
    if "pytest" in stripped or "unittest" in stripped:
        return True
    if re.search(r"(^|\n)\s*!\w+", stripped):
        return True
    return False

def self_contained_enough(code: str) -> bool:
    bad_patterns = [r"pass\s*$", r"TODO", r"FIXME", r"your code here"]
    for pat in bad_patterns:
        if re.search(pat, code, flags=re.IGNORECASE | re.MULTILINE):
            return False
    return True

def normalize_instruction(text: str) -> str:
    text = normalize_whitespace(text)
    text = re.sub(r"^['\"]|['\"]$", "", text)
    text = re.sub(r"^instruction\s*:\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def normalize_lang(value: str) -> str:
    value = str(value or "").strip().lower()
    aliases = {"py": "python", "python3": "python"}
    return aliases.get(value, value)

def write_jsonl(path: Path, rows: List[Dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with jsonlines.open(path, "w") as writer:
        writer.write_all(rows)

def append_jsonl(path: Path, rows: List[Dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with jsonlines.open(path, "a") as writer:
        writer.write_all(rows)

def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")
    rows = []
    with jsonlines.open(path, "r") as reader:
        for row in reader:
            rows.append(row)
    return rows

def save_json(path: Path, obj: Dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def load_json(path: Path) -> Dict[str, Any]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def extract_json_object(text: str) -> Optional[Dict[str, Any]]:
    if not text:
        return None
    text = text.strip()
    try:
        obj = json.loads(text)
        return obj if isinstance(obj, dict) else None
    except Exception:
        pass

    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not match:
        return None

    try:
        obj = json.loads(match.group(0))
        return obj if isinstance(obj, dict) else None
    except Exception:
        return None

def make_instruction_prompt(code: str) -> str:
    return f"""You are generating synthetic Python coding instructions for instruction-tuning.

Your task is to infer a plausible user instruction that could naturally lead to the given Python code solution.

Requirements:
- Produce exactly one Python coding instruction.
- The instruction must be self-contained, realistic, and concise.
- It should describe the programming task, not mention the existence of code, a snippet, or a solution.
- Do not refer to "the code above", "the following function", or similar meta-language.
- Prefer function-level or small utility tasks that fit the given code.
- Avoid unnecessary verbosity, long narratives, licensing language, or formatting instructions.
- Do not include the answer or any code in the instruction.
- Keep the instruction in plain English.

Return ONLY valid JSON with this schema:
{{
  "instruction": "..."
}}

Python code solution:
```python
{code}
```""".strip()

def build_task_request_line(custom_id: str, code: str) -> Dict[str, Any]:
    return {
        "custom_id": custom_id,
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            "model": MODEL_TASKS,
            "temperature": TASK_TEMPERATURE,
            "messages": [
                {
                    "role": "system",
                    "content": "You create concise Python coding instructions and return strict JSON only."
                },
                {
                    "role": "user",
                    "content": make_instruction_prompt(code)
                },
            ],
            "response_format": {"type": "json_object"},
        },
    }

def build_task_requests(filtered_path: Path = FILTERED_SNIPPETS_PATH,
                        output_path: Path = TASK_REQUESTS_PATH) -> int:
    rows = read_jsonl(filtered_path)
    requests = []

    for row in rows:
        requests.append(build_task_request_line(
            custom_id=row["id"],
            code=row["response"],
        ))

    write_jsonl(output_path, requests)
    return len(requests)

def upload_batch_input(jsonl_path: Path) -> str:
    with open(jsonl_path, "rb") as f:
        batch_input_file = client.files.create(
            file=f,
            purpose="batch",
        )
    return batch_input_file.id

def create_batch(input_file_id: str, description: str) -> Dict[str, Any]:
    batch = client.batches.create(
        input_file_id=input_file_id,
        endpoint="/v1/chat/completions",
        completion_window="24h",
        metadata={"description": description},
    )
    return batch.model_dump()

def submit_batch(jsonl_path: Path, meta_path: Path, description: str) -> Dict[str, Any]:
    input_file_id = upload_batch_input(jsonl_path)
    meta = create_batch(input_file_id=input_file_id, description=description)
    save_json(meta_path, meta)
    return meta

def get_batch(meta_path: Path) -> Dict[str, Any]:
    meta = load_json(meta_path)
    batch_id = meta["id"]
    batch = client.batches.retrieve(batch_id)
    latest = batch.model_dump()
    save_json(meta_path, latest)
    return latest

def show_batch_status(meta_path: Path) -> None:
    meta = get_batch(meta_path)
    print("Batch ID:", meta["id"])
    print("Status:", meta["status"])
    print("Created at:", meta.get("created_at"))
    print("In progress at:", meta.get("in_progress_at"))
    print("Completed at:", meta.get("completed_at"))
    print("Request counts:", meta.get("request_counts"))

def download_file(file_id: str, output_path: Path) -> None:
    content = client.files.content(file_id)
    output_path.write_bytes(content.read())

def download_batch_artifacts(meta_path: Path,
                             output_path: Path,
                             error_path: Optional[Path] = None) -> None:
    meta = get_batch(meta_path)

    output_file_id = meta.get("output_file_id")
    error_file_id = meta.get("error_file_id")

    if not output_file_id:
        raise ValueError("Batch has no output_file_id yet. Check the batch status first.")

    download_file(output_file_id, output_path)

    if error_file_id and error_path is not None:
        download_file(error_file_id, error_path)

def parse_chat_batch_line(row: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    try:
        content = row["response"]["body"]["choices"][0]["message"]["content"]
    except Exception:
        return None
    return extract_json_object(content)

def parse_instruction_outputs(output_path: Path = TASK_OUTPUTS_PATH,
                              filtered_path: Path = FILTERED_SNIPPETS_PATH,
                              dataset_path: Path = GENERATED_DATASET_PATH) -> pd.DataFrame:
    if not output_path.exists():
        raise FileNotFoundError(f"Task output file not found: {output_path}")

    snippet_rows = read_jsonl(filtered_path)
    by_id = {row["id"]: row for row in snippet_rows}

    accepted = []
    seen_pairs = set()

    with jsonlines.open(output_path, "r") as reader:
        for row in reader:
            custom_id = row.get("custom_id")
            payload = parse_chat_batch_line(row)
            if not custom_id or not payload:
                continue

            instruction = normalize_instruction(payload.get("instruction", ""))
            if len(instruction) < 20:
                continue

            if any(bad in instruction.lower() for bad in ["the code above", "the snippet above", "this code", "above code"]):
                continue

            source_row = by_id.get(custom_id)
            if not source_row:
                continue

            response = normalize_whitespace(source_row["response"])
            pair_key = sha256_text(instruction + "\n---\n" + response)
            if pair_key in seen_pairs:
                continue
            seen_pairs.add(pair_key)

            accepted.append({
                "id": custom_id,
                "instruction": instruction,
                "response": response,
                "source_dataset": source_row["source_dataset"],
                "metadata": source_row.get("metadata", {}),
            })

    write_jsonl(dataset_path, accepted)
    df = pd.DataFrame(accepted)
    print("Generated instruction-response pairs:", len(df))
    return df


In [ ]:
# =========================
# 6. Load source dataset
# =========================

dataset = load_dataset(
    HF_DATASET_NAME,
    split=HF_SPLIT,
    streaming=USE_STREAMING,
    token=HF_TOKEN,
)

print("Dataset loaded.")
print(f"Streaming: {USE_STREAMING}")


In [ ]:
# =========================
# 7. Extract raw Python code snippets only
# =========================

accepted = []
rejects = []

iterator = dataset if USE_STREAMING else iter(dataset)
progress_total = None if USE_STREAMING else len(dataset)

for i, record in enumerate(tqdm(iterator, total=progress_total)):
    lang = normalize_lang(record.get("lang", ""))
    if lang != LANGUAGE:
        rejects.append({"stage": "raw_filter", "reason": "non_python_lang", "lang": lang})
        continue

    raw_response = record.get("response", "") or record.get("solution", "") or ""
    response = extract_python_code(raw_response)

    if not isinstance(response, str) or not response.strip():
        rejects.append({"stage": "raw_filter", "reason": "empty_response_after_extraction"})
        continue

    accepted.append({
        "id": f"oss_{len(accepted)+1:07d}",
        "response": response,
        "source_dataset": HF_DATASET_NAME,
        "metadata": {
            "lang": lang,
            "raw_index": record.get("raw_index"),
            "index": record.get("index"),
            "seed": record.get("seed"),
            "openai_fingerprint": record.get("openai_fingerprint"),
            "original_instruction": record.get("instruction", "") or record.get("problem", "") or "",
        }
    })

write_jsonl(RAW_SNIPPETS_PATH, accepted)
write_jsonl(REJECTS_PATH, rejects)

print(f"Collected raw Python rows: {len(accepted):,}")
print(f"Saved to: {RAW_SNIPPETS_PATH}")

print()
print("Example extracted response:")
print(accepted[0]["response"][:500] if accepted else "NONE")


In [ ]:
# =========================
# 8. Filter code snippets for training suitability
# =========================

raw_rows = read_jsonl(RAW_SNIPPETS_PATH)

filtered = []
more_rejects = []
reject_counter = {}

for row in tqdm(raw_rows):
    response = row["response"]

    if trivial_or_low_value(response):
        reason = "trivial_or_low_value"
        more_rejects.append({"id": row["id"], "stage": "second_pass", "reason": reason})
        reject_counter[reason] = reject_counter.get(reason, 0) + 1
        continue

    if not self_contained_enough(response):
        reason = "not_self_contained_enough"
        more_rejects.append({"id": row["id"], "stage": "second_pass", "reason": reason})
        reject_counter[reason] = reject_counter.get(reason, 0) + 1
        continue

    if not parseable_python(response):
        reason = "not_parseable_python"
        more_rejects.append({"id": row["id"], "stage": "second_pass", "reason": reason})
        reject_counter[reason] = reject_counter.get(reason, 0) + 1
        continue

    filtered.append(row)

filtered = filtered[:MAX_SNIPPETS_FOR_GENERATION]

write_jsonl(FILTERED_SNIPPETS_PATH, filtered)
append_jsonl(REJECTS_PATH, more_rejects)

print(f"Second-pass filtered rows: {len(filtered):,}")
print(f"Saved to: {FILTERED_SNIPPETS_PATH}")
print("Reject counts:", reject_counter)


## 9. Phase 1: Build and submit the instruction-generation batch

In [ ]:
# =========================
# 10. Build instruction-generation batch input
# =========================

n_task_requests = build_task_requests()
print("Instruction-generation requests written:", n_task_requests)
print("Path:", TASK_REQUESTS_PATH)


In [ ]:
# =========================
# 11. Submit instruction-generation batch
# =========================

task_meta = submit_batch(
    jsonl_path=TASK_REQUESTS_PATH,
    meta_path=TASK_BATCH_META_PATH,
    description="oss-instruct instruction generation from Magicoder snippets",
)
task_meta


In [ ]:
# =========================
# 12. Check batch status
# =========================

show_batch_status(TASK_BATCH_META_PATH)


In [ ]:
# =========================
# 13. Download batch output and parse generated instructions
# =========================

download_batch_artifacts(
    meta_path=TASK_BATCH_META_PATH,
    output_path=TASK_OUTPUTS_PATH,
    error_path=TASK_ERRORS_PATH,
)

dataset_df = parse_instruction_outputs(
    output_path=TASK_OUTPUTS_PATH,
    filtered_path=FILTERED_SNIPPETS_PATH,
    dataset_path=GENERATED_DATASET_PATH,
)

dataset_df.head()


In [ ]:
# =========================
# 14. Coverage after instruction generation
# =========================

generated_rows = read_jsonl(GENERATED_DATASET_PATH)

print("Generated pairs:", len(generated_rows))
print("Target observations:", TARGET_N)
print("Coverage ratio:", round(len(generated_rows) / TARGET_N, 3))


## 15. Phase 2: Final cleanup and export

In [ ]:
# =========================
# 16. Final cleanup and deduplication
# =========================

generated_rows = read_jsonl(GENERATED_DATASET_PATH)

final_rows = []
final_rejects = []
seen_pair_hashes = set()
seen_instruction_hashes = set()

for row in tqdm(generated_rows):
    instruction = normalize_instruction(row["instruction"])
    response = normalize_whitespace(row["response"])

    if not instruction.endswith((".", "?", ":")):
        instruction += "."

    pair_hash = sha256_text(instruction + "\n---\n" + response)
    instr_hash = sha256_text(instruction.lower())

    if pair_hash in seen_pair_hashes:
        final_rejects.append({"id": row["id"], "stage": "final_cleanup", "reason": "duplicate_pair"})
        continue

    if instr_hash in seen_instruction_hashes:
        final_rejects.append({"id": row["id"], "stage": "final_cleanup", "reason": "duplicate_instruction"})
        continue

    seen_pair_hashes.add(pair_hash)
    seen_instruction_hashes.add(instr_hash)

    final_rows.append({
        "instruction": instruction,
        "response": response,
    })

final_rows = final_rows[:TARGET_N]
append_jsonl(REJECTS_PATH, final_rejects)
write_jsonl(FINAL_DATASET_PATH, final_rows)

print(f"Final dataset size: {len(final_rows):,}")
print(f"Saved to: {FINAL_DATASET_PATH}")


In [ ]:
# =========================
# 17. Quick inspection
# =========================

final_rows = read_jsonl(FINAL_DATASET_PATH)

print("Example count:", len(final_rows))
print()

for i in range(min(3, len(final_rows))):
    print("=" * 80)
    print("INSTRUCTION:")
    print(final_rows[i]["instruction"])
    print()
    print("RESPONSE:")
    print(final_rows[i]["response"][:800])
    print()


In [ ]:
# =========================
# 18. Summary stats
# =========================

final_rows = read_jsonl(FINAL_DATASET_PATH)

df = pd.DataFrame(final_rows)
df["instruction_chars"] = df["instruction"].str.len()
df["response_chars"] = df["response"].str.len()
df["response_lines"] = df["response"].str.count("\n") + 1

df[["instruction_chars", "response_chars", "response_lines"]].describe()


In [ ]:
# =========================
# 19. Optional Hugging Face upload prep
# =========================

from datasets import Dataset

final_rows = read_jsonl(FINAL_DATASET_PATH)
hf_ds = Dataset.from_list(final_rows)
hf_ds


In [ ]:
# from google.colab import files

# output_files = [
#     RAW_SNIPPETS_PATH,
#     FILTERED_SNIPPETS_PATH,
#     TASK_REQUESTS_PATH,
#     TASK_OUTPUTS_PATH,
#     GENERATED_DATASET_PATH,
#     FINAL_DATASET_PATH,
#     REJECTS_PATH,
# ]

# for file_path in output_files:
#     if file_path.exists():
#         print(f"Downloading {file_path.name}...")
#         files.download(str(file_path))
#     else:
#         print(f"File not found: {file_path.name}")
